# Databricks/PySpark for Data Analysts/Engineers — Module 1: Introduction to Spark and Databricks

This course is a natural continuation of the Python, SQL/PostgreSQL, and Statistics
courses. There, we worked with data that fit in a single computer's memory. **Apache
Spark** enters the picture when there's so much data (or computation is so expensive)
that one computer isn't enough anymore -- and the work needs to be spread across many
machines at once. **Databricks** is the most popular platform for running Spark in the
cloud, with notebooks very similar to the Jupyter ones you already know.

## Table of contents
1. [What is Apache Spark?](#sec1)
2. [Spark cluster architecture](#sec2)
3. [RDD vs. DataFrame](#sec3)
4. [SparkSession and your first DataFrame](#sec4)
5. [Lazy evaluation: transformations vs. actions](#sec5)
6. [Databricks Free Edition](#sec6)
7. [Summary and exercises](#sec7)


<a id="sec1"></a>
## 1. What is Apache Spark?

**Apache Spark** is a distributed data processing engine -- it can split one large job
(e.g. crunching a billion rows) across dozens or hundreds of computers working in
parallel, and then combine the results into one whole. Key traits:

- **In-memory processing** -- Spark keeps working data in RAM between computation steps,
  instead of writing it to disk after every stage (the way the older Hadoop MapReduce
  did) -- hence the name "Spark" and the big jump in performance.
- **Multi-language** -- the same engine supports Python (PySpark, this course), Scala,
  Java, R, and SQL.
- **One engine, many uses** -- batch processing, SQL, machine learning (MLlib), stream
  processing (Structured Streaming), and graphs -- all in one framework.

Everything you learned about SQL (the SQL course) and working with DataFrames (`pandas`
in the Python course) carries over here almost one-to-one -- PySpark was deliberately
designed to mirror both of those APIs.

> ⚠️ **Spark isn't a replacement for pandas/SQL on small data**
>
> If your data fits comfortably in one computer's memory (millions, not billions, of rows), plain `pandas` or the PostgreSQL from earlier courses are usually FASTER and simpler -- running Spark has its own overhead (starting a cluster, scheduling jobs). Spark shines once data genuinely doesn't fit on one machine, or computation needs to be spread across many machines for speed.

<a id="sec2"></a>
## 2. Spark cluster architecture

A Spark application runs on a **cluster** made up of a few roles:

- **Driver** -- the "conductor" process: this is where your code (the notebook) lives,
  where the execution plan is built, and where gathered results land (e.g. after
  `.collect()`).
- **Executors** -- worker processes on separate machines that actually do the work
  (reading data, filtering, aggregating) on their slice of the data.
- **Cluster manager** -- allocates resources (how many executors, how much memory/CPU)
  for your application. In Databricks, the platform itself handles this -- you don't
  need to configure it by hand.

Data is split into **partitions** and spread across executors -- each executor processes
its partition **in parallel** with the others. We'll come back to how well those
partitions are chosen in Module 9 (optimization).

> ⚡ **Databricks Free Edition = "serverless" compute**
>
> In the free tier of Databricks, you don't have to think about the cluster manager or the number of machines at all -- you get ready-to-use "serverless" compute (no infrastructure to manage), with reasonable limits for learning and experimentation. What you see in the notebook behaves the same regardless of how many machines are actually working behind the scenes.

<a id="sec3"></a>
## 3. RDD vs. DataFrame

Spark's original, lowest-level API is the **RDD** (*Resilient Distributed Dataset*) -- a
distributed collection of Python objects, very flexible, but with no knowledge of the
data's structure (columns, types), so Spark couldn't optimize it well. The **DataFrame**
(added later) is an RDD with a defined schema (columns + types) -- thanks to this, Spark
knows exactly what data it's processing, and can apply a query optimizer (**Catalyst**)
and a faster execution engine (**Tungsten**), much like a database engine optimizes a SQL
query plan.

**In this course we work almost exclusively with DataFrames** -- that's the standard in
modern Spark and what's used day-to-day in Databricks. We mention RDDs only so you
understand what's underneath.

<a id="sec4"></a>
## 4. SparkSession and your first DataFrame

**`SparkSession`** is the entry point to everything in PySpark -- through it you create
DataFrames, run SQL queries, and read and write files. Let's immediately load our shared
dataset for the rest of the course too -- online store data, the exact same schema as the
SQL course (`customers`, `products`, `employees`, `orders`, `order_items`), but now as
Spark DataFrames.

We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("spark_course").getOrCreate()
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


> 🧱 **In Databricks, `spark` already exists**
>
> In a Databricks notebook, the `spark` variable is automatically ready to use -- you don't need to call `SparkSession.builder...` at all. The code above also works there (it will just get the already-running session), so you can safely leave it in every notebook, and it'll behave identically locally and in Databricks.

In [ ]:
customers.printSchema()


In [ ]:
print(f"Number of customers: {customers.count()}")
print(f"Number of orders: {orders.count()}")
print(f"Number of order items: {order_items.count()}")


<a id="sec5"></a>
## 5. Lazy evaluation: transformations vs. actions

This is the single most important mental shift in Spark compared to `pandas`. Operations
on DataFrames fall into two kinds:

- **Transformations** -- e.g. `select`, `filter`, `withColumn`, `groupBy`. They do NOT
  run immediately -- Spark just records a "plan" of what to do (lazy evaluation).
- **Actions** -- e.g. `show()`, `count()`, `collect()`, writing to a file. Only an action
  makes Spark ACTUALLY execute the whole accumulated transformation plan.

This lets Spark look at the ENTIRE chain of transformations at once and optimize it (e.g.
combining several filters, skipping unneeded columns), instead of executing each step in
isolation exactly as you wrote it.

In [ ]:
# A transformation -- nothing is computed yet, Spark just records a plan
plan = customers.filter(customers.segment == "Premium").select("customer_id", "city")
print(type(plan))


In [ ]:
# Only an action actually triggers computation
plan.show(5)


> ⚡ **explain() shows the plan before anything runs**
>
> The `.explain()` method shows a transformation's execution plan WITHOUT running it -- we'll come back to reading these plans in detail in Module 9. Try it now: `plan.explain()`.

<a id="sec6"></a>
## 6. Databricks Free Edition

**Databricks** is the company and platform built by the creators of Apache Spark -- the
most popular way to run Spark in the cloud without managing clusters yourself.
**Databricks Free Edition** (the successor to the old "Community Edition") is a free
tier for learning:

- You sign up with just an email -- **you don't need your own cloud account**
  (AWS/Azure/GCP); Databricks provides ready-made, free infrastructure.
- You get **"serverless"** compute (no manual cluster configuration) and default storage,
  ready to use right after logging in.
- Available to you: notebooks (very similar to the ones we've worked in throughout this
  course), basic Unity Catalog, building simple dashboards, and job scheduling
  (workflows).
- It has reasonable "fair use" limits on compute size -- this is an environment for
  learning and experimentation, not production workloads, but it's advertised as
  perpetually free (no expiration date).

> 🧱 **This notebook vs. real Databricks**
>
> We wrote the code in this course so it runs identically in a plain Jupyter environment (as before) AND in Databricks Free Edition -- for most modules, it's enough to upload the notebook (`.ipynb`) to your workspace and run the cells. Starting with Module 10 (Delta Lake), some code requires a REAL Databricks environment (or downloading extra libraries locally from the internet) -- we'll clearly mark those as "run this in Databricks".

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- what Apache Spark is and when it's worth using (and when it's NOT),
- cluster architecture: driver, executors, cluster manager, partitions,
- the difference between RDD and DataFrame, and why DataFrame is the standard today,
- `SparkSession` as the entry point to PySpark,
- lazy evaluation: transformations build a plan, actions execute it,
- what Databricks Free Edition is and how to use it.

### 📝 Exercise 1: Transformation or action?

For each of the following DataFrame operations, write in a comment whether it's a transformation or an action: `filter()`, `count()`, `select()`, `show()`, `groupBy()`, `collect()`.

<details>
<summary>Show solution</summary>

```python
# filter()   -- transformation
# count()    -- action
# select()   -- transformation
# show()     -- action
# groupBy()  -- transformation (groupBy alone computes nothing yet, only .agg()+action does)
# collect()  -- action
print("See the comments above.")
```

</details>

### 📝 Exercise 2: Schema and row count

Print the schema (`printSchema()`) and the row count (`count()`) for the `products` DataFrame.

<details>
<summary>Show solution</summary>

```python
products.printSchema()
print(f"Number of products: {products.count()}")
```

</details>

### 📝 Exercise 3: Build a plan, but don't run it

Build (WITHOUT calling any action!) a transformation that filters `orders` down to status `'Cancelled'` and selects only the `order_id` and `customer_id` columns. Check the type of the variable holding this plan.

<details>
<summary>Show solution</summary>

```python
cancelled_plan = orders.filter(orders.status == "Cancelled").select("order_id", "customer_id")
print(type(cancelled_plan))
```

</details>

### 📝 Exercise 4: Now run that plan

Call the `show()` action on the plan from exercise 3, to see the first 10 cancelled orders.

<details>
<summary>Show solution</summary>

```python
cancelled_plan.show(10)
```

</details>

> 🔥 **Challenge: count cancelled orders without intermediate actions**
>
> Build ONE chain of transformations (without calling `show()`/`count()` anywhere in the middle) that: (1) joins `orders` with `customers` (we'll cover `join` in detail in Module 4, but try it now -- `orders.join(customers, "customer_id")`), (2) filters down to status `'Cancelled'`, (3) groups by `segment` and counts rows. Call `.explain()` on this chain BEFORE any action, and only then `.show()`.

<details>
<summary>Show solution</summary>

```python
from pyspark.sql import functions as F

challenge_plan = (
    orders.join(customers, "customer_id")
    .filter(orders.status == "Cancelled")
    .groupBy("segment")
    .agg(F.count("*").alias("cancelled_count"))
)
challenge_plan.explain()
challenge_plan.show()
```

The plan shown by explain() combines the filter, the join, and the aggregation into one optimized execution graph -- that's exactly what Catalyst, mentioned in section 3, does for you.

</details>

## What's next?

In **Module 2** we'll dig into the DataFrame API: how to create, filter, select columns
and add new ones -- the foundation all the following modules build on.